# CAVBench — statistical analysis

Reads the CSV files written by `experiments/run_experiment.py` and
`experiments/rollout_sensitivity.py` (folder `results/`) and tests the hypotheses:

| Hypothesis | Comparison | Test |
|---|---|---|
| H1 | UCRA vs each baseline: success, severe conflicts, speed | Fisher's exact test (rates), Mann–Whitney U (speed) |
| H2 | UCRA vs UCRA without uncertainty: collisions per noise level | Fisher's exact test |
| H3 | UCRA vs UCRA without fallback: collisions and speed loss | Fisher's exact test, Mann–Whitney U |
| H4 | Decision time of UCRA ≤ 100 ms | 95th percentile and share of decisions |
| Theorem 1 | Empirical P(max error ≥ γ) vs bound 2K·exp(−2Nγ²) | comparison |

All tests are two-sided at α = 0.05; Holm-corrected p-values are reported alongside, since several comparisons are made.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

RESULTS = Path("../results")
FIG = Path("figures"); FIG.mkdir(exist_ok=True)
ALPHA = 0.05
PROPOSED = "UCRA"
NOISE_ORDER = ["none", "low", "medium", "high"]

def load(name):
    p = RESULTS / f"{name}.csv"
    if not p.exists():
        print(f"(missing {p} — run the experiment first)")
        return None
    df = pd.read_csv(p)
    df["noise"] = pd.Categorical(df["noise"], NOISE_ORDER, ordered=True)
    return df

def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n; d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d; h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return c - h, c + h

def holm(p):
    p = np.asarray(p, float); order = np.argsort(p); m = len(p)
    adj = np.empty(m); running = 0
    for i, idx in enumerate(order):
        running = max(running, (m - i) * p[idx]); adj[idx] = min(running, 1.0)
    return adj

def summary(df):
    g = df.groupby(["scenario", "arbiter", "noise"], observed=True)
    out = g.agg(episodes=("seed", "count"), success=("success", "mean"),
                collision=("collision", "mean"), severe=("any_severe_conflict", "mean"),
                speed=("avg_speed", "mean"), speed_sd=("avg_speed", "std"),
                dec_ms=("decision_ms_mean", "mean"), dec_p95=("decision_ms_p95", "mean"))
    return out.round(3)

def fisher(a, b, col):
    t = [[int(a[col].sum()), int((~a[col].astype(bool)).sum())],
         [int(b[col].sum()), int((~b[col].astype(bool)).sum())]]
    return stats.fisher_exact(t)[1]

def compare(df, other, cols=("success", "collision", "any_severe_conflict")):
    rows = []
    for (sc, nz), d in df.groupby(["scenario", "noise"], observed=True):
        a, b = d[d.arbiter == PROPOSED], d[d.arbiter == other]
        if a.empty or b.empty: continue
        a, b = a.set_index("seed"), b.set_index("seed")
        common = a.index.intersection(b.index); a, b = a.loc[common], b.loc[common]
        r = {"scenario": sc, "noise": nz, "n": len(common)}
        for c in cols:
            r[f"{c}_{PROPOSED}"] = a[c].mean(); r[f"{c}_{other}"] = b[c].mean()
            r[f"p_{c}"] = fisher(a, b, c)
        r[f"speed_{PROPOSED}"] = a.avg_speed.mean(); r[f"speed_{other}"] = b.avg_speed.mean()
        r["p_speed"] = stats.mannwhitneyu(a.avg_speed, b.avg_speed).pvalue
        rows.append(r)
    out = pd.DataFrame(rows)
    pcols = [c for c in out.columns if c.startswith("p_")]
    if len(out):
        flat = holm(out[pcols].to_numpy().ravel()).reshape(len(out), len(pcols))
        for j, c in enumerate(pcols): out[c + "_holm"] = flat[:, j]
    return out.round(4)

## E1 — UCRA vs baselines (H1)

In [ ]:
e1 = load("e1_baselines")
if e1 is not None:
    display(summary(e1))
    for other in sorted(set(e1.arbiter) - {PROPOSED}):
        print(f"\n=== {PROPOSED} vs {other} ===")
        display(compare(e1, other))

In [ ]:
if e1 is not None:
    for sc, d in e1.groupby("scenario"):
        fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
        for ax, col, title in zip(axes, ["success", "collision", "any_severe_conflict"],
                                  ["Success rate", "Collision rate", "Episodes with PET < 1.5 s"]):
            for arb, g in d.groupby("arbiter"):
                agg = g.groupby("noise", observed=True)[col].agg(["sum", "count"])
                p = agg["sum"] / agg["count"]
                lo, hi = zip(*[wilson(k, n) for k, n in zip(agg["sum"], agg["count"])])
                ax.errorbar(range(len(p)), p, yerr=[p - lo, np.array(hi) - p], marker="o", capsize=3, label=arb)
            ax.set_xticks(range(len(NOISE_ORDER))); ax.set_xticklabels(NOISE_ORDER)
            ax.set_title(title); ax.set_ylim(-0.02, 1.02); ax.set_xlabel("noise level")
        axes[0].legend(fontsize=8); fig.suptitle(sc); fig.tight_layout()
        fig.savefig(FIG / f"e1_{sc}.pdf"); plt.show()

## E2 — uncertainty ablation (H2)
H2 expects fewer collisions with the uncertainty input, particularly as noise increases.

In [ ]:
e2 = load("e2_uncertainty_ablation")
if e2 is not None:
    display(summary(e2))
    for other in sorted(set(e2.arbiter) - {PROPOSED}):
        display(compare(e2, other, cols=("collision", "success", "any_severe_conflict")))

## E3 — fallback ablation (H3)
H3 expects fewer collisions with the fallback, with at most 10% speed loss.

In [ ]:
e3 = load("e3_fallback_ablation")
if e3 is not None:
    display(summary(e3))
    for other in sorted(set(e3.arbiter) - {PROPOSED}):
        c = compare(e3, other, cols=("collision", "success", "any_severe_conflict"))
        c["speed_loss_%"] = (100 * (c[f"speed_{other}"] - c[f"speed_{PROPOSED}"]) / c[f"speed_{other}"]).round(2)
        display(c)

## E5 — decision time (H4)
Run `e5_realtime` with `--workers 1` so that timings are not distorted by parallel runs.

In [ ]:
e5 = load("e5_realtime")
if e5 is not None:
    d = e5[e5.arbiter == PROPOSED]
    t = d.groupby(["scenario", "noise"], observed=True).agg(
        mean_ms=("decision_ms_mean", "mean"), p95_ms=("decision_ms_p95", "mean"),
        max_ms=("decision_ms_max", "max"),
        episodes_p95_le_100=("decision_ms_p95", lambda x: (x <= 100).mean()))
    display(t.round(2))

## E4 — rollout sensitivity (Theorem 1)
Empirical probability that the maximum risk-estimation error over the candidates reaches γ, compared with the bound 2K·exp(−2Nγ²).

In [ ]:
e4 = load("e4_rollout_sensitivity")
GAMMA = 0.15
if e4 is not None:
    rows = []
    for n, g in e4.groupby("N"):
        k = int(g.K.max())
        rows.append({"N": n, "situations": g.situation.nunique() * g.noise.nunique(),
                     "mean_max_error": g.max_abs_error.mean(),
                     "p95_max_error": g.max_abs_error.quantile(0.95),
                     f"P(err >= {GAMMA})": (g.max_abs_error >= GAMMA).mean(),
                     "Theorem 1 bound": min(1.0, 2 * k * np.exp(-2 * n * GAMMA**2))})
    t = pd.DataFrame(rows); display(t.round(4))
    fig, ax = plt.subplots(figsize=(5, 3.2))
    ax.plot(t.N, t[f"P(err >= {GAMMA})"], "o-", label="empirical")
    ax.plot(t.N, t["Theorem 1 bound"], "s--", label="Theorem 1 bound")
    ax.set_xscale("log"); ax.set_xlabel("rollouts N"); ax.set_ylabel(f"P(max error ≥ {GAMMA})")
    ax.legend(); fig.tight_layout(); fig.savefig(FIG / "e4_theorem1.pdf"); plt.show()